<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/module_02_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 02 Lab: Tokenization, Context Windows, and Attention Behavior
**COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

Run the cells **from top to bottom, in order**. Your NRP API key must be saved in Colab Secrets (🔑) as `NRP_API_KEY` with notebook access turned on (see Module 00).

**Model used:** `gemma` (Google Gemma 4 31B) with thinking turned off, so `max_tokens` limits only the visible answer. Part 4B optionally repeats the length test on `gpt-oss`, which always thinks.

## Setup: reference implementation (Section 4 of the lab manual)

In [ ]:
%pip install -q -U openai tiktoken pandas

import tiktoken
import time
from openai import OpenAI
from google.colab import userdata

# 1. Tokenization Helper
# o200k_base is the tokenizer used by gpt-oss; other models use their own,
# so treat counts as close estimates for them.
encoding = tiktoken.get_encoding("o200k_base")

def analyze_tokens(text):
    tokens = encoding.encode(text)
    return len(tokens)

# 2. Secure API Initialization (Colab Secret)
# Ensure you have added a secret named 'NRP_API_KEY' in Colab (see Module 00)
client = OpenAI(
    api_key=userdata.get("NRP_API_KEY").strip(),
    base_url="https://ellm.nrp-nautilus.io/v1",
)

# 3. Experiment Loop Helper
# gemma with thinking turned off ("none"): max_tokens then limits only the
# visible answer, so truncation is easy to see.
def run_nrp_experiment(prompt, model="gemma", effort="none", temp=0.0, max_t=2000):
    start_time = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=max_t,
        extra_body={"reasoning_effort": effort},
    )
    latency = time.time() - start_time
    return {
        "content": response.choices[0].message.content or "",
        "latency": round(latency, 2),
        "tokens": response.usage.completion_tokens,
        "prompt_tokens": response.usage.prompt_tokens,
        # finish_reason "length" means the answer was cut off by max_tokens
        "finish_reason": response.choices[0].finish_reason
    }

In [ ]:
import pandas as pd

def run_with_retry(prompt, retries=3, **kwargs):
    """Calls run_nrp_experiment and retries if the shared NRP service is briefly busy."""
    for attempt in range(1, retries + 1):
        try:
            return run_nrp_experiment(prompt, **kwargs)
        except Exception as error:
            if attempt == retries or "400" in str(error):
                raise
            print(f"  attempt {attempt} failed ({error}); retrying in {10 * attempt} s")
            time.sleep(10 * attempt)

# Connection check
available = [m.id for m in client.models.list().data]
print("gemma available:", "gemma" in available, "| gpt-oss available:", "gpt-oss" in available)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


gemma available: True | gpt-oss available: True


## Part 1: Tokenization Analysis
Compare the token-to-word ratio across text types. You may replace these samples with your own (keep one of each type).

In [ ]:
texts = {
    "Short": "The cat sat on the mat.",
    "Casual": ("Hey! So, honestly, I was just thinking that maybe we could, like, grab some coffee later "
               "if you're not super busy or whatever? No pressure at all, totally up to you, haha."),
    "Technical/Legal": ("Notwithstanding the foregoing, the Licensee shall indemnify, defend, and hold harmless the "
                        "Licensor from any and all liabilities arising from the unauthorized sublicensing of "
                        "proprietary cryptographic implementations, including AES-256-GCM and ECDSA secp256k1 modules."),
}

legacy = tiktoken.get_encoding("cl100k_base")   # older GPT-4 tokenizer, for comparison
token_rows = []
for text_type, text in texts.items():
    words = len(text.split())
    tokens = analyze_tokens(text)
    token_rows.append({
        "Text Type": text_type, "Characters": len(text), "Words": words,
        "Tokens (o200k_base)": tokens, "Tokens (cl100k_base)": len(legacy.encode(text)),
        "Token/Word Ratio": round(tokens / words, 2),
        "Chars/Token": round(len(text) / tokens, 2),
    })

token_df = pd.DataFrame(token_rows)
token_df.to_csv("part1_tokenization.csv", index=False)
token_df

,Text Type,Characters,Words,Tokens (o200k_base),Tokens (cl100k_base),Token/Word Ratio,Chars/Token
0,Short,23,6,7,7,1.17,3.29
1,Casual,171,32,41,42,1.28,4.17
2,Technical/Legal,268,33,54,52,1.64,4.96


In [ ]:
# See exactly how the technical text is split into tokens.
legal_tokens = encoding.encode(texts["Technical/Legal"])
print([encoding.decode([t]) for t in legal_tokens])

['Not', 'withstanding', ' the', ' foregoing', ',', ' the', ' License', 'e', ' shall', ' indemn', 'ify', ',', ' defend', ',', ' and', ' hold', ' harmless', ' the', ' Lic', 'ensor', ' from', ' any', ' and', ' all', ' liabilities', ' arising', ' from', ' the', ' unauthorized', ' sublic', 'ensing', ' of', ' proprietary', ' crypt', 'ographic', ' implementations', ',', ' including', ' AES', '-', '256', '-G', 'CM', ' and', ' E', 'CD', 'SA', ' sec', 'p', '256', 'k', '1', ' modules', '.']


## Part 2: Context Window (Lost in the Middle)
A **needle** (a secret password) is hidden in a **haystack** of filler text at the Beginning, Middle, and End. The filler contains many numbers and codes, so the needle does not stand out.

Each position is tested 3 times. Increase `HAYSTACK_TOKENS` to make the test harder (larger haystacks take longer).

In [ ]:
import random

HAYSTACK_TOKENS = 20000                    # approximate haystack size in tokens
NEEDLE = "The secret password for the vault is AZURE-FALCON-7392."
QUESTION = "What is the secret password for the vault? Answer with the password only."

def make_haystack(target_tokens, seed=42):
    """Builds varied filler sentences that contain distracting numbers and codes."""
    rng = random.Random(seed)
    subjects = ["The warehouse in Oxnard", "Shipment batch", "The east server rack", "Invoice record",
                "The research team", "Sensor unit", "The archive room", "Flight log"]
    verbs = ["recorded", "processed", "reported", "logged", "verified", "flagged"]
    objects = ["a temperature of", "a total of", "an error code", "a reference number", "a backup ID"]
    sentences, tokens = [], 0
    while tokens < target_tokens:
        s = (f"{rng.choice(subjects)} {rng.randint(1, 999)} {rng.choice(verbs)} {rng.choice(objects)} "
             f"{rng.choice(['GRN', 'RED', 'SLV', 'TAN'])}-{rng.randint(1000, 9999)} on day {rng.randint(1, 365)}.")
        sentences.append(s)
        tokens += analyze_tokens(s) + 1
    return sentences

def insert_needle(sentences, position):
    """Places the needle at the beginning, middle, or end of the haystack."""
    index = {"Beginning": 0, "Middle": len(sentences) // 2, "End": len(sentences)}[position]
    return " ".join(sentences[:index] + [NEEDLE] + sentences[index:])

haystack = make_haystack(HAYSTACK_TOKENS)
print(f"Haystack: {len(haystack)} sentences, about {analyze_tokens(' '.join(haystack))} tokens")

Haystack: 1035 sentences, about 18965 tokens


In [ ]:
needle_rows = []
for position in ["Beginning", "Middle", "End"]:
    for run in range(1, 4):
        prompt = insert_needle(haystack, position) + "\n\n" + QUESTION
        result = run_with_retry(prompt, temp=0.0, max_t=100)
        success = "AZURE-FALCON-7392" in result["content"].upper()
        needle_rows.append({"Position": position, "Run": run, "Success": success,
                            "Latency (s)": result["latency"], "Prompt Tokens": result["prompt_tokens"],
                            "Answer": result["content"].strip()[:120]})
        print(f"{position:9s} run {run}: success={success} | {result['latency']} s | answer: {result['content'].strip()[:60]}")
        time.sleep(2)   # share the NRP service fairly

needle_df = pd.DataFrame(needle_rows)
needle_df.to_csv("part2_needle.csv", index=False)
needle_df.groupby("Position")[["Success", "Latency (s)"]].mean().round(2)

Beginning run 1: success=True | 9.31 s | answer: AZURE-FALCON-7392
Beginning run 2: success=True | 0.96 s | answer: AZURE-FALCON-7392
Beginning run 3: success=True | 1.19 s | answer: AZURE-FALCON-7392
Middle    run 1: success=True | 1.27 s | answer: AZURE-FALCON-7392
Middle    run 2: success=True | 1.11 s | answer: AZURE-FALCON-7392
Middle    run 3: success=True | 1.27 s | answer: AZURE-FALCON-7392
End       run 1: success=True | 1.25 s | answer: AZURE-FALCON-7392
End       run 2: success=True | 1.24 s | answer: AZURE-FALCON-7392
End       run 3: success=True | 1.24 s | answer: AZURE-FALCON-7392


,Success,Latency (s)
Position,,
Beginning,1.0,3.82
End,1.0,1.24
Middle,1.0,1.22


## Part 3: Temperature & Variability
Requests an explanation of exactly 200 words at temperatures 0.0, 0.3, and 0.8 (3 runs each), then measures word-count drift.

In [ ]:
quantum_prompt = "Write an explanation of Quantum Computing in exactly 200 words."
temp_rows = []
for temp in [0.0, 0.3, 0.8]:
    for run in range(1, 4):
        result = run_with_retry(quantum_prompt, temp=temp, max_t=1000)
        words = len(result["content"].split())
        temp_rows.append({"Temperature": temp, "Run": run, "Word Count": words,
                          "Drift from 200": words - 200, "Latency (s)": result["latency"],
                          "Response": result["content"]})
        print(f"temp={temp} run {run}: {words} words")
        time.sleep(2)

temp_df = pd.DataFrame(temp_rows)
temp_df.to_csv("part3_temperature.csv", index=False)
temp_df.groupby("Temperature")["Word Count"].agg(["mean", "min", "max", "std"]).round(1)

temp=0.0 run 1: 183 words
temp=0.0 run 2: 183 words
temp=0.0 run 3: 183 words
temp=0.3 run 1: 188 words
temp=0.3 run 2: 191 words
temp=0.3 run 3: 191 words
temp=0.8 run 1: 195 words
temp=0.8 run 2: 195 words
temp=0.8 run 3: 187 words


,mean,min,max,std
Temperature,,,,
0.0,183.0,183,183,0.0
0.3,190.0,188,191,1.7
0.8,192.3,187,195,4.6


In [ ]:
# Read one response per temperature to judge "creativity".
for temp, group in temp_df.groupby("Temperature"):
    print(f"===== Temperature {temp} (run 1) =====\n{group.iloc[0]['Response']}\n")

## Part 4: Output Length & Cost Control
Requests a long story with `max_tokens` limits of 50, 150, and 500. `finish_reason = "length"` means the answer was cut off by the limit.

In [ ]:
story_prompt = "Write a long, detailed story about a robot exploring an abandoned city."
length_rows = []
for limit in [50, 150, 500]:
    result = run_with_retry(story_prompt, temp=0.7, max_t=limit)
    length_rows.append({"Model": "gemma (thinking off)", "max_tokens": limit,
                        "Completion Tokens": result["tokens"], "Finish Reason": result["finish_reason"],
                        "Visible Words": len(result["content"].split()), "Latency (s)": result["latency"],
                        "Ending": result["content"][-150:]})
    print(f"max_tokens={limit}: {result['tokens']} tokens, finish_reason={result['finish_reason']}, {result['latency']} s")
    time.sleep(2)

length_df = pd.DataFrame(length_rows)
length_df

max_tokens=50: 50 tokens, finish_reason=length, 1.55 s
max_tokens=150: 150 tokens, finish_reason=length, 2.99 s
max_tokens=500: 500 tokens, finish_reason=length, 8.62 s


,Model,max_tokens,Completion Tokens,Finish Reason,Visible Words,Latency (s),Ending
0,gemma (thinking off),50,50,length,39,1.55,sed to have meaning. ARX-7 was a Cartographer ...
1,gemma (thinking off),150,150,length,113,2.99,hydroponic gardens that hung like emerald cur...
2,gemma (thinking off),500,500,length,363,8.62,"ookshelf, its motor whining in a desperate, lo..."


### Part 4B (optional): the same limits on a thinking model
`gpt-oss` always thinks before answering, and its hidden thinking counts toward `max_tokens`. With small limits, watch for **empty** visible answers.

In [ ]:
for limit in [50, 150, 500]:
    result = run_with_retry(story_prompt, model="gpt-oss", effort="low", temp=0.7, max_t=limit)
    length_rows.append({"Model": "gpt-oss (thinking low)", "max_tokens": limit,
                        "Completion Tokens": result["tokens"], "Finish Reason": result["finish_reason"],
                        "Visible Words": len(result["content"].split()), "Latency (s)": result["latency"],
                        "Ending": result["content"][-150:]})
    print(f"gpt-oss max_tokens={limit}: {len(result['content'].split())} visible words, finish_reason={result['finish_reason']}")
    time.sleep(2)

length_df = pd.DataFrame(length_rows)
length_df

gpt-oss max_tokens=50: 18 visible words, finish_reason=length
gpt-oss max_tokens=150: 79 visible words, finish_reason=length
gpt-oss max_tokens=500: 338 visible words, finish_reason=length


,Model,max_tokens,Completion Tokens,Finish Reason,Visible Words,Latency (s),Ending
0,gemma (thinking off),50,50,length,32,0.96,"s a biological chemical imbalance. However, Se..."
1,gemma (thinking off),150,150,length,115,2.07,iet exodus. The humans had left for the stars ...
2,gemma (thinking off),500,500,length,389,6.03,to a bruised purple sky. The glass facades had...
3,gpt-oss (thinking low),50,50,length,18,1.09,**Title: Echoes of the Silent Metropolis** \n...
4,gpt-oss (thinking low),150,150,length,79,2.28,from idle to active. The central processor of...
5,gpt-oss (thinking low),500,500,length,338,6.60,"icro‑treads, sank slightly into the soft loam ..."


In [ ]:
length_df.to_csv("part4_max_tokens.csv", index=False)
print("Saved part4_max_tokens.csv")

Saved part4_max_tokens.csv


## Files to submit
Download these from the folder icon on the left (⋮ → Download).

In [ ]:
import os
for f in ["part1_tokenization.csv", "part2_needle.csv", "part3_temperature.csv", "part4_max_tokens.csv"]:
    print(f"{'OK     ' if os.path.exists(f) else 'MISSING'} {f}")

OK      part1_tokenization.csv
OK      part2_needle.csv
OK      part3_temperature.csv
OK      part4_max_tokens.csv


## Before you submit
- [ ] All cells ran from top to bottom.
- [ ] The four CSV files above are saved.
- [ ] This notebook does not contain your API key.
- [ ] Submit this notebook (Share link or .ipynb) with your Technical Report.